# 09. Expectations - 汚れたデータをどう扱うか

`08` では、パイプラインが正しく動くことだけを見ていました。
実際のデータは、そうきれいには届きません。金額が負だったり、商品名が空だったりします。

LDPには、そういう行を **条件で宣言して扱う** 仕組みがあります。Expectations (期待) です。

強さが3段階あり、それぞれ **条件を1つだけ書く形** と **まとめて書く形** があります。

| | 違反した行 | 更新 | 条件が1つ | 条件が複数 |
|---|---|---|---|---|
| 警告 | 残す | 続く | `expect` | `expect_all` |
| 除去 | 捨てる | 続く | `expect_or_drop` | `expect_all_or_drop` |
| 停止 | — | **失敗する** | `expect_or_fail` | `expect_all_or_fail` |

`_all` が付くかどうかは **書き方の違いだけ** です。違反したときの扱いは変わりません。
条件を1つずつ書くとデコレータが縦に積み上がるので、同じ扱いのものは辞書でまとめられる、というだけです。
このノートブックでは条件が2つあるので `_all` の形を使います。

このノートブックで確かめること:

1. 同じ条件でも、デコレータが違うと結果がどう変わるか
2. 違反したことをどうやって知るか
3. どれを選ぶか

**前提**: `00_setup` と `08` を終えていること。


## 準備


In [1]:
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.runtime import dbutils

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()
w = WorkspaceClient(profile="free")

In [2]:
CATALOG = "tech_survey"
SCHEMA = "ldp"
LANDING = f"/Volumes/{CATALOG}/ops/landing/09_quality"

# resources/pipelines/09_quality.pipeline.yml で付けた名前
PIPELINE_NAME = "09_quality_ldp"

# パイプラインの動きが記録されるテーブル。4章で使う
EVENT_LOG = f"{CATALOG}.{SCHEMA}.quality_event_log"

## 1. わざと汚れたデータを置く

正常な注文を20件と、条件に引っかかる行を3件用意します。

- `product` が `null` のもの
- `amount` が負のもの
- `amount` が `0` のもの


In [3]:
import json
import random
import uuid

try:
    dbutils.fs.rm(LANDING, True)
except NotFound:
    pass

rows = [
    {
        "order_id": str(uuid.uuid4()),
        "product": random.choice(["laptop", "monitor", "keyboard"]),
        "amount": random.randint(1000, 50000),
    }
    for _ in range(20)
]

# ここから3件はわざと壊してある
rows += [
    {"order_id": str(uuid.uuid4()), "product": None, "amount": 40000},
    {"order_id": str(uuid.uuid4()), "product": "mouse", "amount": -500},
    {"order_id": str(uuid.uuid4()), "product": "monitor", "amount": 0},
]

dbutils.fs.put(f"{LANDING}/orders.json", "\n".join(json.dumps(r) for r in rows), True)
print("置いた行数:", len(rows))

置いた行数: 23


## 2. ソースを読む

ソースは [`src/pipelines/09_quality/transformations/`](../../pipelines/09_quality/transformations) にあります。
`quality_bronze` が取り込み、そこから2つのテーブルを作ります。

**条件は2つとも同じです。**

```python
RULES = {
    "positive_amount": "amount > 0",
    "product_not_null": "product IS NOT NULL",
}
```

**quality_warn.py**

```python
@dp.table(comment="違反を記録するだけで、行はすべて残す")
@dp.expect_all(RULES)
def quality_warn():
    return spark.readStream.table("quality_bronze")
```

**quality_drop.py**

```python
@dp.table(comment="違反した行を落として、きれいなものだけ残す")
@dp.expect_all_or_drop(RULES)   # ← 違うのはここ1行だけ
def quality_drop():
    return spark.readStream.table("quality_bronze")
```

条件はSQLの文字列で書きます。Pythonの関数は使えません。
パイプラインは条件をSQLとして評価するので、Pythonの世界の値は持ち込めない、と考えると分かりやすいです。

なお条件に付けている `positive_amount` のような名前は、後でメトリクスに出てきます。
何が何件違反したかを見分けるためのものなので、**具体的に付けておくと後で困りません。**


## 3. デプロイして動かす

`08` と同じく、ソースを変えたらデプロイが要ります。リポジトリのルートで実行してください。

```sh
databricks bundle deploy
```

3つのテーブルの件数がどうなるか、動かす前に予想してみてください。


In [4]:
PIPELINE_ID = next(
    p.pipeline_id for p in w.pipelines.list_pipelines() if p.name.endswith(PIPELINE_NAME)
)

print(f"{w.config.host}/pipelines/{PIPELINE_ID}")

https://dbc-6f498009-072a.cloud.databricks.com/pipelines/723679c9-3619-4551-9910-ea27e26005ea


In [6]:
import time

from databricks.sdk.service.pipelines import UpdateInfoState

DONE = (UpdateInfoState.COMPLETED, UpdateInfoState.FAILED, UpdateInfoState.CANCELED)


def run_pipeline(full_refresh: bool = False) -> UpdateInfoState:
    """
    パイプラインの更新を起動し、終わるまで待つ。

    Parameters
    ----------
    full_refresh : bool, default False
        True にすると、ストリーミングテーブルも最初から作り直す。

    Returns
    -------
    UpdateInfoState
        更新の最終状態: COMPLETED / FAILED / CANCELED のいずれか。
    """
    update = w.pipelines.start_update(pipeline_id=PIPELINE_ID, full_refresh=full_refresh)

    while True:
        state = w.pipelines.get_update(PIPELINE_ID, update.update_id).update.state
        print(state.value)
        if state in DONE:
            return state
        time.sleep(20)

In [7]:
run_pipeline(full_refresh=True)

CREATED
WAITING_FOR_RESOURCES
INITIALIZING
SETTING_UP_TABLES
COMPLETED


<UpdateInfoState.COMPLETED: 'COMPLETED'>

In [8]:
# 取り込んだ数、警告だけの数、捨てた後の数を並べて見る
for t in ("quality_bronze", "quality_warn", "quality_drop"):
    print(t, spark.table(f"{CATALOG}.{SCHEMA}.{t}").count())

quality_bronze 23
quality_warn 23
quality_drop 20


`quality_warn` は `quality_bronze` と同じ数になったはずです。
条件に違反した行も、**そのままテーブルに入っています。** `expect` は記録するだけだからです。

`quality_drop` だけが減っています。違反した3件が書き込む前に落とされました。

ここで大事なのは、**どちらもエラーになっていない** ことです。
`06` の `txnVersion` や `07` のウォーターマークと同じで、黙って進みます。
違反があったことは、こちらから見に行かないと分かりません。


## 4. 違反をどう知るか

パイプラインの動きは **イベントログ** に記録されます。
`09_quality.pipeline.yml` で、これをUnity Catalogのテーブルに出すよう指定してあります。

```yaml
event_log:
  catalog: tech_survey
  schema: ldp
  name: quality_event_log
```

指定しないとパイプラインの内部に置かれ、SQLから引きにくくなります。
**あとで品質を追いたいなら、最初からテーブルに出しておくのが楽です。**

まず、どんな形をしているのかを見ます。


In [9]:
display(spark.sql(f"DESCRIBE {EVENT_LOG}"))

,col_name,data_type,comment
0,id,string,None
1,sequence,"struct<data_plane_id:struct<instance:string,seq_no:bigint>,control_plane_seq_no:bigint>",None
2,origin,"struct<cloud:string,region:string,org_id:bigint,user_id:bigint,pipeline_id:string,pipeline_type:string,pipeline_name:string,cluster_id:string,update_id:string,maintenance_id:string,table_id:string,table_name:string,flow_id:string,flow_name:string,batch_id:bigint,request_id:string,uc_resource_id:string,dataset_name:string,sink_name:string,catalog_name:string,schema_name:string,materialization_name:string,operation_id:string,flow_attempt_id:string,source_name:string,... 10 more fields>",None
3,timestamp,timestamp,None
4,message,string,None
5,level,string,None
6,maturity_level,string,None
7,error,"struct<fatal:boolean,exceptions:array<struct<class_name:string,message:string,stack:array<struct<declaring_class:string,method_name:string,file_name:string,line_number:bigint>>>>>",None
8,details,string,None
9,event_type,string,None


`details` の中にJSONが入っていて、期待の結果はそこに埋まっています。
`details:flow_progress:data_quality:expectations` の位置です。`:` はJSONの中をたどる演算子です。

配列になっているので、`from_json` で型を与えてから `explode` で行に開きます。


In [10]:
display(
    spark.sql(f"""
        SELECT
            e.dataset,
            e.name AS expectation,
            SUM(e.passed_records) AS passed,
            SUM(e.failed_records) AS failed
        FROM (
            SELECT explode(
                from_json(
                    details:flow_progress:data_quality:expectations,
                    'array<struct<name: string, dataset: string, passed_records: int, failed_records: int>>'
                )
            ) AS e
            FROM {EVENT_LOG}
            WHERE event_type = 'flow_progress'
        )
        GROUP BY 1, 2
        ORDER BY 1, 2
    """)
)

,dataset,expectation,passed,failed
0,tech_survey.ldp.quality_drop,positive_amount,21,2
1,tech_survey.ldp.quality_drop,product_not_null,22,1
2,tech_survey.ldp.quality_warn,positive_amount,21,2
3,tech_survey.ldp.quality_warn,product_not_null,22,1


どのテーブルの、どの条件が、何件通って何件落ちたかが出ます。
`warn` と `drop` で **違反の件数は同じ** はずです。見つけ方は同じで、見つけた後の扱いだけが違うからです。

同じ内容は、3章で出したパイプラインの画面でも見られます。
テーブルをクリックすると、期待ごとの件数がグラフで出ます。
普段見るならそちらが速く、**継続的に監視したりアラートを出すならこのテーブル** を使うことになります。


## 5. `expect_or_fail` を試すなら

3つ目の強さが `expect_or_fail` です。違反を1件でも見つけた時点で **更新全体を失敗させます。**

ここでは動かしていません。動かすと更新が失敗するので、
このノートブックを上から流すと必ず赤くなってしまうからです。

試す場合は `quality_drop.py` のデコレータを1行変えて、デプロイし直します。

```python
@dp.expect_all_or_fail(RULES)
```

```sh
databricks bundle deploy
```

そのうえで `run_pipeline()` を実行すると `FAILED` で止まります。
このとき **テーブルは更新されません。違反した行だけでなく、正常な行も1件も入りません。**
「一部だけ入って、残りは入っていない」という中途半端な状態を作らないのが、この強さの意味です。


## 6. どれを選ぶか

| | 違反した行 | 更新 | 使いどころ |
|---|---|---|---|
| `expect` | 残す | 続く | まず現状を測りたい。止めるほどではない |
| `expect_or_drop` | 捨てる | 続く | 下流に流したくない。捨ててよいと判断できる |
| `expect_or_fail` | — | 失敗する | 壊れたまま進むほうが危ない。請求、在庫など |

迷ったら **`expect` (警告のみ) から始める** のが安全です。

いきなり `drop` にすると想定より多くの行が消えますし、
`fail` にすると夜中にパイプラインが止まります。
まず警告で流して違反件数を測り、「この条件は本当に守られるべきか」
「違反は何件くらい出るのか」を見てから強くする、という順序になります。

層で考えるなら、こうなります。

- **Bronze** … 何も付けない。届いたものをそのまま受ける層なので、ここで弾くと元データが失われる
- **Silver** … `drop` で整える。下流が前提にしてよい状態を作る
- **業務上あり得ない値だけ** `fail`。「金額が負の請求」のような、進むほうが危ないもの


## 考えてみる

- `expect_or_drop` で捨てられた行を、後から中身まで見ることはできますか
- Bronze に `expect_or_fail` を付けるのは、よい考えでしょうか
- `08` で調べた REPLACE WHERE フローと、expectations は併用できましたか


### 答え

**Q1. 捨てられた行の中身**

**見られません。** テーブルには入らず、イベントログにも件数しか残りません。

中身が要るなら、方法は2つあります。

- **Bronze を残しておいて、そこから条件で引く。** 今回の構成ならこれができます
- **`expect` (警告) にしておいて、条件を反転させた別テーブルを作る。**
  違反した行だけを集めた置き場 (隔離テーブル) を用意する形です

「捨てる」は後から取り返せない操作なので、
捨てる前の層を残しておくかどうかを先に決めておく必要があります。

**Q2. Bronze に `fail` を付けるか**

**よくありません。** Bronze は「届いたものをそのまま受ける」層です。

ここで止めると、上流が壊れた瞬間に取り込み自体が止まります。
その間に届いたデータは、元が消えていれば戻せません。
壊れたデータでも **まず受けておいて**、使う側の Silver で判断するほうが安全です。

品質チェックは「捨てる場所」ではなく「判断する場所」に置く、と考えると迷いません。

**Q3. REPLACE WHERE との併用**

**できません。** `08` で調べた制約のとおりです。

REPLACE WHERE フローを張ったテーブルには expectations を付けられません。
「範囲を絞ってやり直せること」と「品質チェック」のどちらを取るか、という選択になります。

その場合は、品質チェックを別の層に分けることになります。
REPLACE WHERE で作り直すテーブルの手前か後ろに、チェック用のテーブルを1つ挟む形です。


## 後片付け

このノートブックで作ったものを消したいときだけ、コメントを外して実行します。


In [ ]:
# for t in ("quality_drop", "quality_warn", "quality_bronze"):
#     spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.{t}")
# dbutils.fs.rm(LANDING, True)